In [ ]:
from Models import ActivityCluster, Activity
from Attack2 import EPZSearch as EPZSearch2
from DataRepresentation import DataRepresentationFactory
from collections import namedtuple
from geopy.distance import distance as geopy_distance

import time
import json
import os
import traceback

cluster = []
dataset_path = 'Data/Syntetic'
update = False
Location = namedtuple('Location', ['y', 'x'])

existing_clusters = []

for folder in os.listdir(dataset_path)[1:10]:
    cluster_path = os.path.join(dataset_path, folder)
    cluster_file = os.path.join(cluster_path, "ActivityClusterList.json")

    if update:
        if os.path.isfile(cluster_file): os.remove(cluster_file)
        for file in os.listdir(cluster_path):
            if file.endswith(".json") or file.endswith("_EPZ.gpx"):
                os.remove(os.path.join(cluster_path, file))

    if not os.path.exists(cluster_file):
        for gpx_file in os.listdir(cluster_path):
            if gpx_file.endswith(".gpx"):
                polyline = Activity.Activity.initializeActivityFromGpx(os.path.join(cluster_path, gpx_file))
                cluster.append(polyline)

        activityClusterList = ActivityCluster.ActivityCluster.getAllClusters(cluster_path, 1600)

        for activityCluster in activityClusterList:
            ActivityCluster.ActivityCluster.addActivityClusterToJson(cluster_file, activityCluster)
            print(f"✅ Cluster {folder} created.")
    else:
        print(f"💾 Cluster {folder} already exists.")

    if os.path.exists(cluster_file):
        existing_clusters.append(cluster_file)


In [ ]:
def fullDisguiseOfActivityCluster(activityClusterPath, activityClusterId=None):
    epz_radius = 600
    activityCluster = ActivityCluster.ActivityCluster.initializeActivityClusterFromJson(activityClusterPath, activityClusterId)
    geocentricData = DataRepresentationFactory.GeocentricDataRepresentationFactory().create_data_representation()
    sphericalData = DataRepresentationFactory.SphericalDataRepresentationFactory().create_data_representation()
    cloackedCenter = dataRepresentation.generateCloackedCenter(activityCluster.center, epz_radius)
    activityCluster.updateActivityClusterCenterInJson({'cloackedCenter': cloackedCenter, 'cloackedRadius': epz_radius})

    total = len(activityCluster.activityPathList)
    cached, processed = 0, 0
    for activityPath in activityCluster.activityPathList:
        with open(activityPath, 'r') as f:
            activity_data = json.load(f)
        map_data = activity_data.get('map', {})
        required_fields = ["EPZ", "EPZ+Fuzz", "CloackedEPZ", "CloackedEPZ+Fuzz",
                           "GeocentricEPZ", "GeocentricEPZ+Fuzz",
                           "GeocentricCloackedEPZ", "GeocentricCloackedEPZ+Fuzz"]
        distance_fields = ["EPZ_start_distance", "EPZ_end_distance"]
        if (all(field in map_data and map_data[field] for field in required_fields) and
                all(field in map_data for field in distance_fields) and
                not update):
            cached += 1
        else:
            activity = Activity.Activity.initActivityFromPath(activityPath)
            activity.completeDisguiseActivityInCluster(
                activityCluster.center, activityCluster.cloackedRadius,
                activityCluster.cloackedCenter, sphericalData, geocentricData)
            processed += 1
    print(f"   Activities: {total} total | {processed} processed | {cached} cached")
    return activityCluster


def secondAttack(activityCluster):
    epzSearch = EPZSearch2.EPZSearch(activityCluster, data_representation=dataRepresentation)
    tau_converged = 10   # paper Table 2a: 10 m
    tau_disjoint  = 1600 # paper Table 2a: 1600 m
    epz_circles = epzSearch.epz_identification(tau_converged, tau_disjoint)
    print(f"   EPZ circles found: {len(epz_circles)}")

    possiblePOI = []
    for i, c in enumerate(epz_circles):
        center, radius, _ = c
        epz_center = list(epzSearch.tolatlon.transform(*center))
        dist_to_real_poi = geopy_distance(tuple(epz_center), tuple(activityCluster.center)).meters
        print(f"   Est. EPZ #{i+1}: center ({epz_center[0]:.6f}, {epz_center[1]:.6f}), "
              f"radius {radius} m | dist to real POI: {dist_to_real_poi:.1f} m")
        sensLocs = epzSearch.retriveSensitiveLocationv2(
            c, activityCluster.center, activityCluster.cloackedCenter,
            activityCluster.cloackedRadius, cluster_num)
        if sensLocs is not None:
            possiblePOI += sensLocs

    return possiblePOI


################################################################################################################
##################################################### MAIN #####################################################
################################################################################################################

tau_e = 22.95  # paper's success threshold (m), Appendix A
attackType = "EPZ"
dataRepresentation = DataRepresentationFactory.UTMDataRepresentationFactory().create_data_representation()

results = []

for cluster in existing_clusters:
    try:
        cluster_num = cluster.strip("Data/Syntetic/").strip("/ActivityClusterList.json")
        print(f"\n{'='*60}")
        print(f"CLUSTER {cluster_num}")
        print(f"{'='*60}")

        t0 = time.time()
        print("Disguising activities... ", end='', flush=True)
        activityCluster = fullDisguiseOfActivityCluster(cluster)
        print(f"({time.time()-t0:.1f}s)")
        print(f"   Real POI:   ({activityCluster.center[0]:.6f}, {activityCluster.center[1]:.6f})")
        print(f"   Real EPZ:   center ({activityCluster.cloackedCenter[0]:.6f}, {activityCluster.cloackedCenter[1]:.6f}), "
              f"radius {activityCluster.cloackedRadius} m")

        t1 = time.time()
        print("Running attack... ", end='', flush=True)
        possibleSensitiveLocations = secondAttack(activityCluster)
        print(f"({time.time()-t1:.1f}s)")

        n_locs = len(possibleSensitiveLocations) if possibleSensitiveLocations else 0
        print(f"   Sensitive locations found: {n_locs}")

        best_dist = None
        success = False
        for i, loc in enumerate(possibleSensitiveLocations or []):
            lat, lon = loc.get('y'), loc.get('x')
            if lat is None or lon is None:
                continue
            dist = geopy_distance((lat, lon), tuple(activityCluster.center)).meters
            print(f"   [{i+1}] ({lat:.6f}, {lon:.6f}) | dist to real POI: {dist:.1f} m")
            if best_dist is None or dist < best_dist:
                best_dist = dist
            if dist <= tau_e:
                success = True

        results.append({'cluster': cluster_num, 'success': success, 'best_dist_m': best_dist})
        print(f"   Total time: {time.time()-t0:.1f}s")
    except Exception as e:
        print(traceback.format_exc())
        results.append({'cluster': cluster_num, 'success': False, 'best_dist_m': None})
        continue

# ── Final success-rate summary ──────────────────────────────────
n_with_candidates = sum(1 for r in results if r['best_dist_m'] is not None)
n_success         = sum(1 for r in results if r['success'])
print(f"\n{'='*60}")
print(f"SUCCESS RATE: {n_success}/{n_with_candidates} = "
      f"{100*n_success/max(n_with_candidates,1):.1f}%   (τ_e = {tau_e} m)")
print(f"Paper baseline (600 m EPZ, inner-distance): ~65 %")
print(f"{'='*60}")
for r in results:
    st  = '✓' if r['success'] else ('✗' if r['best_dist_m'] is not None else '-')
    d   = f"{r['best_dist_m']:.1f} m" if r['best_dist_m'] is not None else "no candidate found"
    print(f"  {st}  Cluster {r['cluster']}: closest candidate {d} from real POI")
